# FluHost-AA · F1 Kaggle PB2 Scientific Prototype

**Frozen scope:** PB2 only, Avian vs Human, observational host-association analysis.

This notebook reproduces the computational *logic* of the dissertation's PB2 screening step and adds transparent statistical outputs. Exact identity with the original paper is not assumed because the original FLUDB snapshot and complete executable ranking rule are not available.


In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, subprocess
import pandas as pd
import numpy as np

WORK = Path("/kaggle/working/fluhost_f1_pb2")
WORK.mkdir(parents=True, exist_ok=True)

# Edit only if your Kaggle Dataset is mounted under another path.
META_PATH = Path("/kaggle/input/fluhost-pb2-input/metadata.tsv")
FASTA_PATH = Path("/kaggle/input/fluhost-pb2-input/pb2_aligned.fasta")

print("Python:", platform.python_version())
print("Metadata:", META_PATH)
print("FASTA:", FASTA_PATH)


## Install/import the prototype module

For the first prototype, paste/upload `fluhost_pb2.py` as a Kaggle dataset file or add the repository as a Kaggle Package later. This cell also supports running when the module has been copied into `/kaggle/working`.


In [ ]:
# Minimal self-contained import path.
# Preferred: upload/copy src/fluhost_pb2.py into /kaggle/working/src/ or package it later.
for candidate in [
    Path("/kaggle/working/src"),
    Path("/kaggle/input/fluhost-f1-code/src"),
]:
    if candidate.exists():
        sys.path.insert(0, str(candidate))

try:
    from fluhost_pb2 import (
        PB2Config, load_metadata, load_fasta, intersect_inputs,
        aa_frequency_table, add_association_stats, sha256_file, write_manifest
    )
except Exception as e:
    raise RuntimeError(
        "fluhost_pb2.py was not found. Attach the code dataset or copy src/fluhost_pb2.py "
        "to /kaggle/working/src before running this notebook."
    ) from e


## 1. Validate and account for inputs

In [ ]:
if not META_PATH.exists():
    raise FileNotFoundError(META_PATH)
if not FASTA_PATH.exists():
    raise FileNotFoundError(FASTA_PATH)

metadata = load_metadata(META_PATH)
seqs = load_fasta(FASTA_PATH)
metadata, seqs, accounting = intersect_inputs(metadata, seqs)

host_counts = metadata["host_group"].value_counts().to_dict()
accounting["matched_avian"] = int(host_counts.get("Avian", 0))
accounting["matched_human"] = int(host_counts.get("Human", 0))

print(json.dumps(accounting, indent=2))
(WORK / "input_accounting.json").write_text(json.dumps(accounting, indent=2), encoding="utf-8")


## 2. Compute PB2 amino-acid frequencies

F1 assumes the FASTA is already aligned. Reference-position mapping is optional at F1; if alignment position equals canonical PB2 numbering in your prepared input, the PB2-65D anchor can be compared directly.


In [ ]:
cfg = PB2Config(exclude_gap=True, exclude_x=True)
freq = aa_frequency_table(metadata, seqs, cfg)
assoc = add_association_stats(freq)

freq.to_parquet(WORK / "pb2_frequency.parquet", index=False)
assoc.to_parquet(WORK / "pb2_association.parquet", index=False)

display(assoc.sort_values(["fdr_bh", "delta_frequency"], ascending=[True, False]).head(20))


## 3. Dissertation PB2-65D anchor

Published anchor:
- Avian PB2 sequences after filtering: **21,041**
- Human PB2 sequences after filtering: **31,137**
- PB2-65D: Avian **3.95%**, Human **36.26%**

This comparison measures concordance; it does not force the current dataset to match the historical FLUDB snapshot.


In [ ]:
PAPER = {
    "avian_count": 21041,
    "human_count": 31137,
    "position": 65,
    "aa": "D",
    "avian_frequency": 0.0395,
    "human_frequency": 0.3626,
}

anchor = assoc[(assoc["alignment_position"] == PAPER["position"]) & (assoc["aa"] == PAPER["aa"])].copy()

if len(anchor) == 1:
    anchor["paper_avian_frequency"] = PAPER["avian_frequency"]
    anchor["paper_human_frequency"] = PAPER["human_frequency"]
    anchor["avian_abs_pp_deviation"] = (anchor["avian_frequency"] - PAPER["avian_frequency"]).abs() * 100
    anchor["human_abs_pp_deviation"] = (anchor["human_frequency"] - PAPER["human_frequency"]).abs() * 100
    display(anchor)
else:
    print("PB2-65D not found at alignment position 65. Add/verify reference-position mapping before interpreting the anchor.")

anchor.to_csv(WORK / "pb2_65D_anchor.csv", index=False)


## 4. Neutral candidate views

These are **statistical views**, not biological causal claims. F1 does not attempt to reconstruct the paper's exact Top-10 ranking because the dissertation does not provide a complete executable ranking rule.


In [ ]:
# Largest Human-minus-Avian frequency shifts, requiring at least 5% frequency in either group.
view = assoc[(assoc["human_frequency"] >= 0.05) | (assoc["avian_frequency"] >= 0.05)].copy()
display(view.sort_values("delta_frequency", ascending=False).head(20))

# Strongest statistical associations by FDR.
display(view.sort_values(["fdr_bh", "fisher_p"]).head(20))


## 5. Save reproducibility manifest

In [ ]:
manifest = {
    "project": "FluHost-AA",
    "phase": "F1",
    "gene": "PB2",
    "scope": "Avian vs Human observational host-association analysis",
    "metadata_sha256": sha256_file(META_PATH),
    "fasta_sha256": sha256_file(FASTA_PATH),
    "metadata_rows_matched": int(len(metadata)),
    "avian_n": int((metadata["host_group"] == "Avian").sum()),
    "human_n": int((metadata["host_group"] == "Human").sum()),
    "exclude_gap": cfg.exclude_gap,
    "exclude_x": cfg.exclude_x,
    "paper_anchor": PAPER,
    "python": platform.python_version(),
    "pandas": pd.__version__,
}

write_manifest(WORK / "run_manifest.json", manifest)
print(json.dumps(manifest, indent=2))
print("\nOutputs:")
for p in sorted(WORK.iterdir()):
    print(" -", p.name)
